#  PDS - Project - Analysis of Statistical Similarity of Network Services
**Author**: Matúš Remeň (xremen01)\
**Academic year**: 2023/24

---
# Cluster Analysis

## Dataset Preparation
Contains the loading of the dataset, cleaning, and feature engineering.

In [ ]:
# Import libraries.
import pandas as pd

pd.set_option("display.float_format", "{:.8f}".format)

# Define constants.
INPUT_FILE_PATH = "tls-pds-07-03-2024.parquet"

# Load dataset - network flows - from an Apache Parquet file.
df: pd.DataFrame = pd.read_parquet(INPUT_FILE_PATH)

# Remove flows which were not finished properly.
df = df[df["uint8 TCP_FLAGS"] & 0b0000_0001 == 1]

# Rewrite some TLS_SNI values which contain some ID/hash to reduce the number of unique values.
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.safeframe.googlesyndication.com", value="safeframe.googlesyndication.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.googlevideo.com", value="googlevideo.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.ingest.sentry.io", value="ingest.sentry.io", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.c.2mdn.net", value="c.2mdn.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.fls.doubleclick.net", value="fls.doubleclick.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.metric.gstatic.com", value="metric.gstatic.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.c.drive.google.com", value="c.drive.google.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.cloudfront.net", value="cloudfront.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.ampproject.net", value="ampproject.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.eshop-rychle.cz", value="eshop-rychle.cz", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.myshopify.com", value="myshopify.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.webpkgcache.com", value="webpkgcache.com", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace="collector.*.px-cloud.net", value="collector.px-cloud.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace="collector.*.perimeterx.net", value="collector.perimeterx.net", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.prmutv.co", value="prmutv.co", regex=True)
df["string TLS_SNI"] = df["string TLS_SNI"].replace(to_replace=".*.tapecontent.net", value="tapecontent.net", regex=True)

# Engineering new features.
if "time TIME_FIRST" in df.columns:  # Workaround for re-running the cell.
    # Flow duration in ns.
    df["float FLOW_DURATION"] = (df["time TIME_LAST"] - df["time TIME_FIRST"]).dt.total_seconds()
    # Mean packet size.
    df["float AVG_PACKET_SIZE"] = df["uint64 BYTES"] / df["uint32 PACKETS"]
    # Mean packet size reverse.
    df["float AVG_PACKET_SIZE_REV"] = df["uint64 BYTES_REV"] / df["uint32 PACKETS_REV"]
    # Number of packets.
    df["int TOTAL_PACKETS"] = df["uint32 PACKETS"] + df["uint32 PACKETS_REV"]
    # Number of bytes.
    df["int TOTAL_BYTES"] = df["uint64 BYTES"] + df["uint64 BYTES_REV"]
    # Time between packets.
    df["float TIME_BETWEEN_PACKETS"] = df["float FLOW_DURATION"] / df["int TOTAL_PACKETS"]
    # Download/upload speed. Convert time to seconds.
    df["float TRAFFIC_SPEED"] = df["int TOTAL_BYTES"] / df["float FLOW_DURATION"]
    # Download/Upload ratio (<1 download / >1 upload is major).
    df["float TRAFFIC_RATIO"] = df["uint32 PACKETS"] / df["uint32 PACKETS_REV"]

    # Drop columns, which might not be needed anymore.
    df = df.drop(columns=["time TIME_FIRST", "time TIME_LAST", "int8* PPI_PKT_DIRECTIONS", "time* PPI_PKT_TIMES", "uint16* PPI_PKT_LENGTHS", "uint8* PPI_PKT_FLAGS"])

## Clustering
Firstly, the dataset values are normalized using the logarithm function, similarly as in the first notebook.
Then, the dataset is reduced to 8 components using PCA.
Finally, the algorithm KMeans is utilized to cluster the data into 4 clusters.

The other clustering methods - DBSCAN, Hierarchical Clustering - were tried, but were very slow, and had high memory consumption.

Per checks performed on the YouTube videos, the domain `googlevideo.com` provided video streaming, while `www.youtube.com` was providing
the page html, and JavaScript. Domains `dns.google` and `doh.opendns.com` are DNS-over-HTTPS services. The configuration of the parameters
described above leads to clustering, which indicates their difference, although because of noisy data, the results are not perfectly precise.
The results are visualized in the following cells.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np


tls_sni_column = df["string TLS_SNI"]
df_norm = df.drop(columns=["string TLS_SNI"])
for col in df_norm.columns:
    df_norm[col] = np.log10(df_norm[col] + 1)

In [ ]:
COMPONENTS = 8
pc = PCA(n_components=COMPONENTS).fit_transform(df_norm)
df_pca = pd.DataFrame(data=pc, columns=[f"{i}" for i in range(COMPONENTS)])

In [ ]:
CLUSTERS = 4
kmeans = KMeans(n_clusters=CLUSTERS, n_init=32)
labels = kmeans.fit_predict(df_pca)

In [ ]:
labeled_df = pd.DataFrame({
    "TLS_SNI": tls_sni_column,
    "Cluster": labels,
})

### Interesting TLS_SNI distribution visualization

In [ ]:
_, axs = plt.subplots(2, 2, figsize=(12, 12))
axs = axs.flatten()

domains = ["googlevideo.com", "www.youtube.com", "dns.google", "doh.opendns.com"]
for i, domain in enumerate(domains):
    counts = labeled_df[labeled_df["TLS_SNI"] == domain]["Cluster"].value_counts()
    total_count = df[df["string TLS_SNI"] == domain].shape[0]
    counts.plot.pie(ax=axs[i], autopct="%1.1f%%")
    axs[i].set_title(f"Cluster distribution for {domain}")
    axs[i].set_ylabel('')
    axs[i].set_xlabel(f"Total: {total_count}")


### Cluster visualization

In [ ]:
_, axs = plt.subplots(2, 2, figsize=(12, 12))
axs = axs.flatten()

for i in range(CLUSTERS):
    counts = labeled_df[labeled_df['Cluster'] == i]['TLS_SNI'].value_counts().head(8)
    counts.plot.pie(ax=axs[i], autopct='%1.1f%%')
    axs[i].set_title(f'TLS_SNI distribution for Cluster {i}')
    axs[i].set_ylabel('')
    total_count = labeled_df[labeled_df['Cluster'] == i].shape[0]
    axs[i].set_xlabel(f"Total: {total_count}")

plt.tight_layout()
plt.show()